In [1]:
import pandas as pd
from datasets import load_dataset
import os
import numpy as np
import ast

# --- Configuration ---
DATASET_URL = "FronkonGames/steam-games-dataset"
LOCAL_CLEANED_FILE = "steam_games_cleaned.csv"
DATASET_SPLIT = "train"

# --- 1. Load the Dataset ---
print(f"Attempting to load dataset: {DATASET_URL}...")

try:
    dataset_dict = load_dataset(DATASET_URL)
    split = DATASET_SPLIT if DATASET_SPLIT in dataset_dict else list(dataset_dict.keys())[0]
    df = dataset_dict[split].to_pandas()
    print(f"Dataset loaded successfully from '{split}' split.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    if os.path.exists(LOCAL_CLEANED_FILE):
        print(f"Loading existing cleaned file: {LOCAL_CLEANED_FILE}")
        df = pd.read_csv(LOCAL_CLEANED_FILE)
    else:
        raise SystemExit("Cannot continue: dataset not found and no local backup file.")

print(f"Data loaded: {df.shape[0]} rows X {df.shape[1]} columns")


c:\Users\User\anaconda3\envs\steam_ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Attempting to load dataset: FronkonGames/steam-games-dataset...


Dataset loaded successfully from 'train' split.
Data loaded: 83560 rows X 39 columns


### 資料核心欄位說明
| 欄位 | 說明 | 用途 |
|------|------|------|
| **Name** | 遊戲名稱 | 作為唯一識別與顯示名稱；在報表與圖表中標示用。 |
| **Release date** | 發行日期 | 可轉換成年份或月份，用於觀察發行趨勢與時間對評價／銷量的影響。 |
| **Estimated owners** | 預估擁有者（字串區間） | 可解析為中位數或範圍估計，作為「銷量 proxy」指標。 |
| **Peak CCU** | 昨日同時上線人數 | 反映近期熱度與活躍度，可用於熱門遊戲趨勢分析。 |
| **Required_age** | 年齡限制 | 分析不同分級遊戲的評價與市場表現差異。 |
| **Price** | 售價（美元） | 用於研究免費遊戲與付費遊戲在銷量、評價、黏著度上的差異。 |
| **DLC count** | DLC（追加下載內容）數量 | 評估遊戲後續內容延伸策略及其對評價／銷量的影響。 |
| **Supported languages** | 支援語言列表 | 反映遊戲的國際化程度，可分析語言覆蓋對銷量的關聯。 |
| **Full audio languages** | 支援語音語言列表 | 反映遊戲的國際化程度，可分析語音語言覆蓋對銷量的關聯。 |
| **Windows / Mac / Linux** | 支援作業系統 | 用於平台支援分析，探討多平台支援是否帶來更高銷量或評價。 |
| **Metacritic score** | 專業媒體評分 | 作為品質指標之一，代表專業評論者的評價。 |
| **User score** | 玩家用戶評分 | 反映一般玩家的主觀滿意度，可與專業評分對照。 |
| **Positive / Negative** | 好評與負評數量 | 可計算「好評率」作為遊戲整體評價指標。 |
| **Achievements** | 成就數 | 衡量遊戲深度與內容豐富度，可分析是否與玩家黏著度相關。 |
| **Recommendations** | 被推薦次數 | 作為人氣與互動量指標，常與評分或銷量正相關。 |
| **Average playtime forever** | 平均總遊玩時數（分鐘） | 代表長期黏著度，越高表示玩家持續遊玩、內容豐富。 |
| **Average playtime two weeks** | 平均最近兩週遊玩時數（分鐘） | 反映近期熱度，可用於短期人氣與活躍度分析。 |
| **Median playtime forever** | 玩家中位數總遊玩時數（分鐘） | 減少極端值干擾，更能反映一般玩家的真實投入時間。 |
| **Median playtime two weeks** | 玩家中位數最近兩週遊玩時數（分鐘） | 衡量遊戲在近期的穩定活躍程度。 |
| **Developers / Publishers** | 開發商與發行商 | 分析大型公司與獨立開發團隊的市場表現與評價差異。 |
| **Genres** | 遊戲類型（如 Action, RPG, Simulation） | 主要的分類依據，用於群組分析與偏好探索。 |
| **Categories** | 類別（如 Single-player, Multi-player, VR） | 可用於玩家玩法偏好與平台支援類別分析。 |

### 核心欄位分類
| 分類 | 欄位 | 原因／用途 |
|------|------|-------------|
| **基本資訊** | `Name`, `Release date`, `Required age`, `Price` | 基本屬性與時間分析用；可研究價格、分級與年份趨勢。 |
| **銷量／熱度** | `Estimated owners`, `Peak CCU`, `Recommendations` | 作為銷量或人氣 proxy；分析遊戲成功的市場因素。 |
| **相容性與覆蓋範圍** | `Supported languages`, `Full audio languages`, `Windows`, `Mac`, `Linux` | 探討語言與平台覆蓋度對市場與評價的影響。 |
| **評分相關** | `Metacritic score`, `User score`, `Positive`, `Negative` | 建立品質指標與市場反應關係（專業 vs 玩家）。 |
| **玩家行為** | `Average playtime forever`, `Average playtime two weeks`, `Median playtime forever`, `Median playtime two weeks` | 衡量長期與短期黏著度、遊戲內容深度與熱度變化。 |
| **內容延伸** | `DLC count`, `Achievements` | 評估內容持續性與成就設計對玩家黏著度的影響。 |
| **公司屬性** | `Developers`, `Publishers` | 分析大型公司與獨立開發商的市場差異。 |
| **遊戲分類** | `Genres`, `Categories` | 作為群組分析基礎；探索不同類型與玩法的市場表現。 |

In [2]:
# ============================================
# 資料前處理
# ============================================

# 僅保留分析所需的核心欄位
core_columns = [
    "Name", "Release date", "Estimated owners", "Peak CCU", "Required age", "Price",
    "DLC count", "Supported languages", "Full audio languages",
    "Windows", "Mac", "Linux",
    "Metacritic score", "User score", "Positive", "Negative", "Achievements",
    "Recommendations", "Average playtime forever", "Average playtime two weeks",
    "Median playtime forever", "Median playtime two weeks",
    "Developers", "Publishers", "Categories", "Genres"
]
df = df[core_columns].copy()

# 刪除無 Name 的資料
df = df.dropna(subset=["Name"])

# 填補文字缺值
fill_values = {
    "Developers": "Unknown Developer",
    "Publishers": "Unknown Publisher",
    "Categories": "Uncategorized",
    "Genres": "Unknown Genre"
}
df = df.fillna(value=fill_values)

# 將 Release date 轉為 datetime，並取出年份與月份
df["Release date"] = pd.to_datetime(df["Release date"], errors="coerce")
df["release_year"] = df["Release date"].dt.year
df["release_month"] = df["Release date"].dt.month

# 處理 Estimated owners，取中位數作為銷量 proxy
def parse_owners(x):
    try:
        low, high = x.replace(",", "").split(" - ")
        return (int(low) + int(high)) // 2
    except:
        return np.nan

df["owners_mid"] = df["Estimated owners"].apply(parse_owners)

# 移除重複遊戲名稱
df = df.drop_duplicates(subset=["Name"])

df.to_csv("steam_games_cleaned.csv", index=False, encoding="utf-8-sig")
print(f"清理完成，共 {df.shape[0]} 筆資料、{df.shape[1]} 欄")


清理完成，共 82842 筆資料、29 欄


In [3]:
# ============================================
# 特徵工程
# ============================================

# 好評率
df["positive_rate"] = df["Positive"] / (df["Positive"] + df["Negative"])
df["positive_rate"] = df["positive_rate"].fillna(0)

# 平台支援數
df["platform_count"] = df[["Windows", "Mac", "Linux"]].sum(axis=1)

# 支援語言數量
df["lang_count"] = df["Supported languages"].apply(
    lambda x: len(str(x).split(",")) if pd.notnull(x) else 0
)

# 支援語音語言數量
def count_audio_languages(x):
    if pd.isna(x):
        return 0
    if isinstance(x, list):
        return len(x)
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return len(parsed)
            else:
                return len(x.split(","))
        except:
            return len(x.split(","))
    return 0

df["audio_lang_count"] = df["Full audio languages"].apply(count_audio_languages)

# 語音支援比例
df["audio_lang_ratio"] = df["audio_lang_count"] / (df["lang_count"] + 1)

# 近期 vs 總遊玩時數比
df["playtime_ratio_recent"] = df["Average playtime two weeks"] / (df["Average playtime forever"] + 1)

# 總評論數與評價極性
df["review_count"] = df["Positive"] + df["Negative"]
df["review_log"] = np.log1p(df["review_count"])
df["review_balance"] = (df["Positive"] - df["Negative"]) / (df["Positive"] + df["Negative"] + 1)

# 價格與銷量對數化
df["price_log"] = np.log1p(df["Price"])
df["owners_log"] = np.log1p(df["owners_mid"])

# 免費與否標記
df["is_free"] = (df["Price"] == 0).astype(int)

# 主要類別與主要類型
df["main_genre"] = df["Genres"].apply(lambda x: str(x).split(",")[0] if pd.notnull(x) else "Unknown")
df["main_category"] = df["Categories"].apply(lambda x: str(x).split(",")[0] if pd.notnull(x) else "Unknown")

# 評價落差（玩家 vs 媒體）
df["rating_gap"] = df["User score"] - df["Metacritic score"]

# 年代與新舊遊戲標記
df["release_decade"] = (df["release_year"] // 10) * 10
df["is_recent"] = (df["release_year"] >= 2020).astype(int)

# ============================================
# 儲存清理後的資料
# ============================================

df.to_csv("steam_games_feature.csv", index=False, encoding="utf-8-sig")
print(f"特徵工程完成，共 {df.shape[0]} 筆資料、{df.shape[1]} 欄")


特徵工程完成，共 82842 筆資料、46 欄


In [4]:
df.describe(include='all')

,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DLC count,Supported languages,Full audio languages,Windows,...,review_log,review_balance,price_log,owners_log,is_free,main_genre,main_category,rating_gap,release_decade,is_recent
count,82842,82713,82842,82842.000000,82842.000000,82842.000000,82842.000000,82842,82842,82842,...,82842.000000,82842.000000,82842.000000,82842.000000,82842.000000,82842,82842,82842.000000,82713.000000,82842.000000
unique,82842,NaN,14,NaN,NaN,NaN,NaN,11098,2197,2,...,NaN,NaN,NaN,NaN,NaN,28,26,NaN,NaN,NaN
top,Galactic Bowling,NaN,0 - 20000,NaN,NaN,NaN,NaN,['English'],[],True,...,NaN,NaN,NaN,NaN,NaN,Action,Single-player,NaN,NaN,NaN
freq,1,NaN,54046,NaN,NaN,NaN,NaN,42407,48145,82812,...,NaN,NaN,NaN,NaN,NaN,32510,74423,NaN,NaN,NaN
mean,NaN,2020-04-06 14:52:56.293932288,NaN,133.873361,0.308237,7.199594,0.518095,NaN,NaN,NaN,...,2.722895,0.343337,1.560769,8.523601,0.192282,NaN,NaN,-3.319138,2015.938607,0.601337
min,NaN,1997-06-30 00:00:00,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,...,0.000000,-0.962963,0.000000,0.000000,0.000000,NaN,NaN,-97.000000,1990.000000,0.000000
25%,NaN,2018-06-14 00:00:00,NaN,0.000000,0.000000,0.990000,0.000000,NaN,NaN,NaN,...,0.693147,0.000000,0.688135,9.210440,0.000000,NaN,NaN,0.000000,2010.000000,0.000000
50%,NaN,2020-11-23 00:00:00,NaN,0.000000,0.000000,4.610000,0.000000,NaN,NaN,NaN,...,2.397895,0.395797,1.724551,9.210440,0.000000,NaN,NaN,0.000000,2020.000000,1.000000
75%,NaN,2022-07-18 00:00:00,NaN,1.000000,0.000000,9.990000,0.000000,NaN,NaN,NaN,...,4.174387,0.703162,2.396986,9.210440,0.000000,NaN,NaN,0.000000,2020.000000,1.000000
max,NaN,2025-04-14 00:00:00,NaN,872138.000000,21.000000,999.980000,2366.000000,NaN,NaN,NaN,...,15.692086,0.997788,6.908735,18.826146,1.000000,NaN,NaN,100.000000,2020.000000,1.000000
